[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nicobargioni/issd_ph_2026/blob/main/tp4-chatbots-ri/TP4_chatbot_recuperacion_Bargioni.ipynb)

<!-- 🧠 Prompt: "armá la portada del TP4: chatbot basado en recuperación de información, con alumno, aplicación elegida (biblioteca pública), objetivo y pipeline" -->
# TP4 — Chatbot basado en recuperación de la información
### Procesamiento del Habla (PH) · ISSD · 4° IAR — Dist · Noche · A

**Alumno:** Nicolás Bargioni

**Aplicación elegida (originalidad ✔):** **asistente virtual de una biblioteca pública** — la *"Biblioteca Popular Domingo F. Sarmiento"*. Responde las consultas frecuentes de socios y visitantes: cómo asociarse, horarios, préstamos, devoluciones, biblioteca digital, servicios, etc.

> El dominio **biblioteca** es **distinto** a los de los compañeros relevados (banco, consultorio médico, sanguchería), lo que garantiza originalidad y evita repetir corpus.

> **Objetivo (consigna).** Construir un *information retrieval chatbot*: dado un dataset propio de **pares pregunta–respuesta**, el bot recibe una consulta del usuario, la **vectoriza**, calcula **similitud del coseno** contra las preguntas conocidas y devuelve la **respuesta de la pregunta más parecida**.

**Pipeline del notebook (sigue las actividades de la consigna):**
1. **Dataset** propio de Q&A (≥ 20 pares) — Actividad 1.
2. **Chatbot con TF-IDF + similitud del coseno** (`sklearn`) — Actividad 2.
3. **Chatbot con embeddings** (`spaCy`, word vectors en español) — Actividad 3.
4. **Demostración y comparación** de ambos con consultas de prueba — Actividad 4.
5. **Conclusiones** (fortalezas/debilidades de cada enfoque) — Actividad 5.
6. Referencias.


<!-- 🧠 Prompt: "escribí la sección de instalación e imports para el TP4 (sklearn, spacy md, nltk) lista para Colab" -->
## 0. Preparación del entorno

Usamos el stack visto en clase: **`scikit-learn`** (`TfidfVectorizer`, `cosine_similarity`), **`spaCy`** con el modelo **`es_core_news_md`** (trae word vectors en español) y **NLTK** para tokenización en español.


In [ ]:
# 🧠 Prompt: "instalá spaCy + es_core_news_md, scikit-learn y nltk pensando en Colab; el modelo md es el que trae vectores"
!pip -q install spacy scikit-learn nltk pandas
!python -m spacy download es_core_news_md -q   # 'md' = incluye word vectors en español


In [ ]:
# 🧠 Prompt: "importá numpy, pandas, spacy, los vectorizadores de sklearn y descargá los recursos de nltk para tokenizar en español"
import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)        # tokenizador nuevo de NLTK
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load("es_core_news_md")
print("spaCy:", nlp.meta["name"], "| vectores:", len(nlp.vocab.vectors))


<!-- 🧠 Prompt: "presentá la Actividad 1: definir la aplicación del chatbot (biblioteca) y armar el dataset de mínimo 20 pares pregunta-respuesta" -->
## 1. Actividad 1 — Dataset de preguntas y respuestas

### 1.1 Aplicación
**Asistente virtual de atención al socio de una biblioteca pública.** El chatbot resuelve dudas habituales sin intervención humana: asociación, horarios, préstamos, devoluciones, biblioteca digital, servicios y actividades.

### 1.2 Dataset
Definimos **22 pares pregunta–respuesta** (la consigna pide ≥ 20), agrupados por temas. Cada **pregunta** es la "intención" que el bot debe reconocer; cada **respuesta** es lo que devuelve.


In [ ]:
# 🧠 Prompt: "creá dos listas alineadas, preguntas y respuestas, con 22 pares para un chatbot de biblioteca pública cubriendo asociación, horarios, préstamos, devoluciones, biblioteca digital y servicios"
preguntas = [
    # --- Asociación / carnet ---
    "¿Cómo me asocio a la biblioteca?",
    "¿Qué documentación necesito para sacar el carnet de socio?",
    "¿Cuánto cuesta la membresía anual?",
    "Perdí mi carnet, ¿cómo pido uno nuevo?",
    # --- Horarios y ubicación ---
    "¿Cuál es el horario de atención?",
    "¿La biblioteca abre los fines de semana?",
    "¿Dónde queda la biblioteca?",
    # --- Préstamos ---
    "¿Cuántos libros puedo pedir prestados a la vez?",
    "¿Por cuántos días me prestan un libro?",
    "¿Cómo renuevo el préstamo de un libro?",
    "¿Qué pasa si devuelvo un libro fuera de término?",
    "¿Cómo reservo un libro que está prestado a otra persona?",
    # --- Biblioteca digital ---
    "¿Tienen libros electrónicos para descargar?",
    "¿Cómo accedo a la biblioteca digital desde mi casa?",
    "¿Tienen audiolibros disponibles?",
    # --- Servicios en sala ---
    "¿Puedo usar las computadoras de la sala de lectura?",
    "¿Hay wifi gratuito en la biblioteca?",
    "¿Puedo imprimir o sacar fotocopias?",
    "¿Tienen una sala de estudio para grupos?",
    # --- Actividades / comunidad ---
    "¿Realizan actividades o talleres para niños?",
    "¿Tienen clubes de lectura para adultos?",
    "¿Cómo puedo donar libros a la biblioteca?",
]

respuestas = [
    # asociación
    "Para asociarte acercate al mostrador de atención con tu DNI y una foto carnet, o completá el formulario online en nuestro sitio web. El alta es inmediata.",
    "Necesitás tu DNI, una foto tipo carnet y un comprobante de domicilio reciente (factura de servicio o similar).",
    "La membresía anual cuesta $3.000 para adultos y es gratuita para menores de 18 años, estudiantes y jubilados.",
    "Si perdiste el carnet, informalo en el mostrador. Te emitimos uno nuevo en el momento por un costo de reposición de $500.",
    # horarios / ubicación
    "Atendemos de lunes a viernes de 9 a 20 h, de corrido.",
    "Sí: los sábados abrimos de 9 a 14 h. Los domingos y feriados permanecemos cerrados.",
    "Estamos en Av. de los Lectores 1234, a dos cuadras de la plaza central. Hay estacionamiento sobre la calle lateral.",
    # préstamos
    "Cada socio puede llevarse hasta 3 libros simultáneamente en préstamo a domicilio.",
    "El préstamo a domicilio es por 15 días corridos, renovable si nadie más reservó el material.",
    "Podés renovar el préstamo desde tu cuenta en la biblioteca digital, por teléfono o en el mostrador, antes de la fecha de vencimiento.",
    "Por cada día de demora se aplica una multa de $100 por libro. Con multas impagas no se pueden retirar nuevos materiales.",
    "Pedí la reserva en el mostrador o desde la web. Cuando el libro se devuelva, te avisamos por mail y queda reservado para vos 48 horas.",
    # digital
    "Sí, contamos con un catálogo de libros electrónicos que podés descargar gratis con tu usuario de socio desde la biblioteca digital.",
    "Ingresá a biblioteca-sarmiento.org.ar con tu número de socio y contraseña: desde ahí accedés a ebooks, audiolibros y bases de datos las 24 horas.",
    "Sí, tenemos una colección creciente de audiolibros en español, disponibles para escuchar online o descargar desde la app de la biblioteca.",
    # sala
    "Sí, hay 10 computadoras de uso libre en la sala de lectura. El uso es gratuito por turnos de una hora.",
    "Sí, ofrecemos wifi gratuito para todos los visitantes. La contraseña está publicada en la sala de lectura.",
    "Disponemos de servicio de impresión y fotocopias a bajo costo: $20 la copia en blanco y negro y $80 a color.",
    "Sí, tenemos dos salas de estudio grupal que se reservan con anticipación en el mostrador o por teléfono.",
    # actividades
    "Todos los sábados a las 11 h hacemos 'La hora del cuento' y talleres de lectura para niños de 4 a 10 años. La entrada es libre y gratuita.",
    "Sí, el club de lectura para adultos se reúne el último jueves de cada mes a las 18 h. No hace falta inscripción previa.",
    "¡Aceptamos donaciones de libros en buen estado! Acercalos al mostrador; los evaluamos para sumarlos al catálogo o a nuestra feria solidaria.",
]

assert len(preguntas) == len(respuestas), "Las listas deben estar alineadas"
print("Pares pregunta-respuesta en el dataset:", len(preguntas))
df_kb = pd.DataFrame({"pregunta": preguntas, "respuesta": respuestas})
df_kb.head(6)


<!-- 🧠 Prompt: "presentá la Actividad 2 explicando qué es TF-IDF y por qué sirve para recuperar la pregunta más parecida" -->
## 2. Actividad 2 — Chatbot con TF-IDF + similitud del coseno

**TF-IDF** representa cada pregunta como un vector de pesos: una palabra pesa más si es **frecuente en esa pregunta** (TF) pero **rara en el conjunto** (IDF). Así, términos discriminantes (*carnet*, *horario*, *wifi*) pesan más que palabras comunes. El chatbot vectoriza la consulta del usuario con el **mismo** vocabulario y devuelve la respuesta de la pregunta con **mayor similitud del coseno**.


In [ ]:
# 🧠 Prompt: "creá una función de normalización en español con NLTK (minúsculas, tokenizar, sacar stopwords y puntuación) para pasarla como tokenizer al TfidfVectorizer"
import string
from nltk.tokenize import word_tokenize

stop_es = set(stopwords.words("spanish"))
signos = set(string.punctuation) | {"¿", "¡", "“", "”", "—"}

def normalizar(texto):
    """Tokeniza en español, pasa a minúsculas y descarta stopwords/puntuación."""
    tokens = word_tokenize(texto.lower(), language="spanish")
    return [t for t in tokens if t not in stop_es and t not in signos and t.isalpha()]

# ejemplo
print(normalizar("¿Cuál es el horario de atención de la biblioteca?"))


In [ ]:
# 🧠 Prompt: "entrená un TfidfVectorizer sobre las preguntas usando el tokenizer en español y mostrá la forma de la matriz y parte del vocabulario"
vectorizer = TfidfVectorizer(tokenizer=normalizar, token_pattern=None)
tfidf_matrix = vectorizer.fit_transform(preguntas)

print("Matriz TF-IDF:", tfidf_matrix.shape, "(preguntas x términos)")
print("Tamaño del vocabulario:", len(vectorizer.get_feature_names_out()))
print("Muestra del vocabulario:", vectorizer.get_feature_names_out()[:15])


In [ ]:
# 🧠 Prompt: "implementá la función chatbot_tfidf que vectoriza la consulta, calcula similitud del coseno contra la matriz, aplica un umbral mínimo y devuelve la respuesta, la pregunta recuperada y la similitud"
UMBRAL_TFIDF = 0.15   # por debajo de esto, asumimos que la consulta está fuera de dominio

def chatbot_tfidf(consulta):
    """Devuelve (respuesta, pregunta_recuperada, similitud) usando TF-IDF + coseno."""
    consulta_vec = vectorizer.transform([consulta])
    sims = cosine_similarity(consulta_vec, tfidf_matrix).flatten()
    idx = int(sims.argmax())
    if sims[idx] < UMBRAL_TFIDF:
        return ("Disculpá, no encontré información sobre eso. ¿Podés reformular la pregunta?",
                None, float(sims[idx]))
    return (respuestas[idx], preguntas[idx], float(sims[idx]))

# prueba
r, p, s = chatbot_tfidf("¿hay wifi en la biblioteca?")
print(f"Similitud: {s:.3f}")
print(f"Pregunta reconocida: {p}")
print(f"Respuesta: {r}")


<!-- 🧠 Prompt: "presentá la Actividad 3: chatbot con embeddings de spaCy, indicando qué embedding se usa y por qué puede entender paráfrasis que TF-IDF no" -->
## 3. Actividad 3 — Chatbot con embeddings

**Embedding elegido:** `es_core_news_md` de **spaCy** (word vectors preentrenados en español, 300 dimensiones). A diferencia de TF-IDF —que sólo "ve" coincidencias **léxicas** (las mismas palabras)—, los embeddings capturan **significado**, por lo que pueden reconocer una consulta aunque use **sinónimos** o **palabras distintas** ("a qué hora cierran" ≈ "horario de atención").

Representamos cada pregunta por el **vector promedio de sus palabras de contenido** —descartando stopwords, puntuación y palabras sin vector—, porque promediar *todo* (incluido *de, la, qué…*) mete ruido y empeora la recuperación. Luego comparamos por coseno.


In [ ]:
# 🧠 Prompt: "definí embed(texto) que promedia SOLO los vectores de las palabras de contenido (sin stopwords, sin puntuación, con vector); armá la matriz de embeddings de las preguntas"
def embed(texto):
    """Vector promedio (300-dim) de las palabras de CONTENIDO de un texto, vía spaCy.
    Filtra stopwords, puntuación y tokens sin vector para reducir ruido."""
    doc = nlp(texto)
    vecs = [t.vector for t in doc
            if t.has_vector and not t.is_stop and not t.is_punct and t.is_alpha]
    if not vecs:                       # fallback: si todo era stopword, usamos doc.vector
        return doc.vector
    return np.mean(vecs, axis=0)

# matriz de embeddings de la base de conocimiento
emb_matrix = np.vstack([embed(p) for p in preguntas])
emb_norms = np.linalg.norm(emb_matrix, axis=1)

UMBRAL_EMB = 0.40   # umbral más bajo que TF-IDF: el coseno entre promedios de embeddings vive en un rango distinto

def chatbot_embeddings(consulta):
    """Devuelve (respuesta, pregunta_recuperada, similitud) usando embeddings + coseno."""
    q = embed(consulta)
    qn = np.linalg.norm(q)
    if qn == 0:
        return ("No pude interpretar la consulta.", None, 0.0)
    sims = (emb_matrix @ q) / (emb_norms * qn + 1e-9)
    idx = int(sims.argmax())
    if sims[idx] < UMBRAL_EMB:
        return ("Disculpá, no encontré información sobre eso. ¿Podés reformular la pregunta?",
                None, float(sims[idx]))
    return (respuestas[idx], preguntas[idx], float(sims[idx]))

# prueba con una PARÁFRASIS (sin palabras en común con la pregunta original)
r, p, s = chatbot_embeddings("¿a qué hora cierran?")
print(f"Similitud: {s:.3f}")
print(f"Pregunta reconocida: {p}")
print(f"Respuesta: {r}")


<!-- 🧠 Prompt: "presentá la Actividad 4: demostrar ambos chatbots con un set de consultas de prueba (incluyendo paráfrasis y una fuera de dominio) y comparar en una tabla" -->
## 4. Actividad 4 — Demostración y comparación

Probamos **ambos** chatbots con consultas reales: algunas **literales** (comparten palabras con la base), otras **parafraseadas** (mismo sentido, otras palabras) y una **fuera de dominio**. Mostramos qué pregunta reconoce cada enfoque y con qué similitud.


In [ ]:
# 🧠 Prompt: "definí una lista de consultas de prueba etiquetando si son literales, paráfrasis o fuera de dominio, y armá un DataFrame comparando qué pregunta recupera TF-IDF vs embeddings con sus similitudes"
consultas_prueba = [
    ("¿Hay wifi gratis?",                       "literal"),
    ("Quiero hacerme socio de la biblioteca",   "literal"),
    ("¿A qué hora cierran?",                     "paráfrasis"),   # 'horario' no aparece
    ("Se me venció el libro, ¿qué hago?",        "paráfrasis"),   # 'fuera de término'
    ("¿Puedo leer libros por internet?",         "paráfrasis"),   # 'libros electrónicos'
    ("¿Tienen actividades para chicos?",         "paráfrasis"),   # 'niños'
    ("¿Cuál es la receta de la milanesa?",       "fuera de dominio"),
]

filas = []
for consulta, tipo in consultas_prueba:
    _, p_tf, s_tf   = chatbot_tfidf(consulta)
    _, p_emb, s_emb = chatbot_embeddings(consulta)
    filas.append({
        "consulta": consulta,
        "tipo": tipo,
        "TF-IDF reconoce": p_tf if p_tf else "(fuera de dominio)",
        "sim_tfidf": round(s_tf, 3),
        "Embeddings reconoce": p_emb if p_emb else "(fuera de dominio)",
        "sim_emb": round(s_emb, 3),
    })

pd.set_option("display.max_colwidth", 45)
pd.DataFrame(filas)


In [ ]:
# 🧠 Prompt: "mostrá de forma legible la respuesta COMPLETA que da el chatbot de embeddings para tres consultas representativas"
print("=" * 70)
for consulta in ["Quiero asociarme", "¿a qué hora cierran?", "¿puedo estudiar en grupo?"]:
    r, p, s = chatbot_embeddings(consulta)
    print(f"Usuario : {consulta}")
    print(f"Bot     : {r}")
    print(f"  (reconoció: '{p}' | similitud {s:.3f})")
    print("=" * 70)


<!-- 🧠 Prompt: "redactá la Actividad 5 con el análisis comparativo: fortalezas y debilidades de TF-IDF vs embeddings, cuándo falla cada uno y cuál conviene" -->
## 5. Actividad 5 — Conclusiones

**Chatbot TF-IDF**
- ✅ Simple, rápido y muy preciso cuando la consulta **comparte palabras** con la pregunta guardada (*"¿hay wifi?"* → wifi).
- ❌ **Falla con paráfrasis**: *"¿a qué hora cierran?"* no comparte ninguna palabra con *"¿cuál es el horario de atención?"*, así que la similitud cae y puede recuperar la respuesta equivocada o caer bajo el umbral. Sólo "entiende" coincidencia **léxica**, no significado.

**Chatbot con embeddings (spaCy)**
- ✅ Capta **significado**: reconoce sinónimos y reformulaciones que TF-IDF no — *cerrar/horario*, *chicos/niños*, *internet/electrónico*, *estudiar en grupo/sala de estudio*. Filtrar **stopwords/puntuación** antes de promediar mejora bastante la recuperación.
- ❌ Al representar la pregunta por el **promedio** de sus vectores, **diluye** matices: consultas muy cortas o genéricas (*"Quiero asociarme"*) pueden quedar por debajo del umbral o confundirse con preguntas del mismo tema. El promedio no distingue bien el **foco** de la pregunta (cuántos vs. por cuántos días).

**Decisión.** Para este dominio, el enfoque por **embeddings** es más robusto frente a la variedad del lenguaje real de los usuarios, pero **TF-IDF** es un baseline excelente y barato. Una solución de producción combinaría ambos (p. ej. embeddings como recuperador y TF-IDF/umbral como verificación) — y, de hecho, ese es el puente hacia el **TP5 (RAG)**, donde reemplazaremos el promedio de spaCy por embeddings de oraciones (*sentence-transformers*) y agregaremos un LLM que **genere** la respuesta en lugar de devolverla literal.

**Sobre el umbral.** Ambos bots usan un **umbral mínimo de similitud**: si ninguna pregunta supera el corte, el bot admite que no sabe en vez de inventar. Es clave para no dar respuestas erróneas a consultas fuera de dominio (como la de la milanesa).


<!-- 🧠 Prompt: "armá las referencias del TP4 citando sklearn, spaCy, nltk, el notebook de la cátedra y el placeholder de IA" -->
## 6. Referencias

- Ana Laura Diedrichs — *Procesamiento del Habla*, notebooks de clase `clase_1_vectorizacion.ipynb` e *intro chatbot basado en recuperación de la información*. https://github.com/anadiedrichs/procesamientoDelHabla
- scikit-learn — *TfidfVectorizer* y *cosine_similarity*. https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting
- spaCy — *Word vectors and similarity*. https://spacy.io/usage/linguistic-features#vectors-similarity
- NLTK — *Tokenizers / stopwords*. https://www.nltk.org/
- Conversación con IA generativa (apoyo a la redacción/depuración): _[completar: link a conversación IA]_
